In [ ]:
import os
from pyspark.sql import SparkSession

os.environ["HADOOP_CONF_DIR"] = "/etc/hadoop/conf"
os.environ["YARN_CONF_DIR"] = "/etc/hadoop/conf"
os.environ["HIVE_CONF_DIR"] = "/etc/hive/conf"

spark = SparkSession.builder \
    .appName("team17 job recommendation modeling") \
    .master("yarn") \
    .config("spark.sql.catalogImplementation", "hive") \
    .config("spark.sql.warehouse.dir", "/user/team17/project/hive/warehouse") \
    .enableHiveSupport() \
    .getOrCreate()

In [2]:
spark.sql("SHOW DATABASES").show()
spark.sql("SHOW TABLES IN team17_projectdb").show()

26/05/09 13:34:05 WARN HiveConf: HiveConf of name hive.metastore.event.db.notification.api.auth does not exist
26/05/09 13:34:06 WARN HiveClientImpl: Detected HiveConf hive.execution.engine is 'tez' and will be reset to 'mr' to disable useless hive logic


+--------------------+
|           namespace|
+--------------------+
|             default|
|             retake1|
|             root_db|
|                show|
|    team00_projectdb|
|          team0_dbms|
|     team0_projectdb|
|             team0db|
|            team0db1|
|            team0db2|
|    team11_projectdb|
|           team12_db|
|team12_hive_proje...|
|    team12_projectdb|
|    team13_projectdb|
|team13_projectdb_...|
|    team14_projectdb|
|    team15_projectdb|
|    team16_projectdb|
|    team17_projectdb|
+--------------------+
only showing top 20 rows

+----------------+-----------------+-----------+
|       namespace|        tableName|isTemporary|
+----------------+-----------------+-----------+
|team17_projectdb|             apps|      false|
|team17_projectdb|        apps_part|      false|
|team17_projectdb|         apps_raw|      false|
|team17_projectdb|             jobs|      false|
|team17_projectdb|        jobs_part|      false|
|team17_projectdb|         job

In [3]:
apps = spark.table("team17_projectdb.apps_part")
jobs = spark.table("team17_projectdb.jobs_part")
users = spark.table("team17_projectdb.users_part")
user_history = spark.table("team17_projectdb.user_history_part")
window_dates = spark.table("team17_projectdb.window_dates")

apps.show(5)
jobs.show(5)
users.show(5)
user_history.show(5)
window_dates.show(5)

+------+-------+-------------------+------+-----+--------+
| appid| userid|               date| jobid|split|windowid|
+------+-------+-------------------+------+-----+--------+
|353565|1471869|2012-04-01 14:20:09|655840|Train|       1|
|353564|1471869|2012-04-01 14:04:36|861987|Train|       1|
|353546|1471791|2012-04-05 14:18:40|766725|Train|       1|
|353414|1471266|2012-04-09 15:22:18|792301|Train|       1|
|353378|1470818|2012-04-04 10:24:20|324128|Train|       1|
+------+-------+-------------------+------+-----+--------+
only showing top 5 rows



+------+--------------------+--------------------+--------------------+----------------+-----+-------+-------+-------------------+-------------------+--------+
| jobid|               title|         description|        requirements|            city|state|country|zipcode|              start|             finish|windowid|
+------+--------------------+--------------------+--------------------+----------------+-----+-------+-------+-------------------+-------------------+--------+
|895739|Outside Sales Rep...|<p><b><span>Outsi...|<p><b><span>Posit...|          Dallas|   TX|     US|   NULL|2012-03-12 11:45:24|2012-04-11 23:59:59|       1|
|900013|             MANAGER|<b>    <b>MANAGER...|Please refer to t...|         Orlando|   FL|     US|  32801|2012-03-05 14:06:58|2012-04-04 23:59:59|       1|
|895719|Exciting Franchis...|Farr's Fresh Ice ...|Farr's Fresh Ice ...|         Orlando|   FL|     US|  32827|2012-03-09 08:26:28|2012-04-08 23:59:59|       1|
|895704|Sales / Customer ...|<p style="t

In [4]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

seed = 42
negative_per_user = 3

positive = apps.select("userid", "jobid", "windowid", "split") \
    .dropDuplicates() \
    .withColumn("label", F.lit(1.0))

positive.show(5)

+-------+------+--------+-----+-----+
| userid| jobid|windowid|split|label|
+-------+------+--------+-----+-----+
|1463694|760794|       1|Train|  1.0|
|1462650|732210|       1|Train|  1.0|
|1459197|376551|       1|Train|  1.0|
|1456526|916652|       1|Train|  1.0|
|1451204|565349|       1|Train|  1.0|
+-------+------+--------+-----+-----+
only showing top 5 rows



In [5]:
jobs_for_negative = jobs.select("jobid", "windowid").dropDuplicates()

job_window = Window.partitionBy("windowid").orderBy(F.rand(seed))

jobs_indexed = jobs_for_negative.withColumn(
    "job_rank",
    F.row_number().over(job_window)
)

job_counts = jobs_indexed.groupBy("windowid") \
    .agg(F.max("job_rank").alias("job_count"))

users_for_negative = positive.select("userid", "windowid", "split").dropDuplicates()

negative_numbers = spark.range(1, negative_per_user + 1) \
    .withColumnRenamed("id", "negative_number")

negative_seed = users_for_negative.crossJoin(negative_numbers) \
    .join(job_counts, "windowid") \
    .withColumn(
        "job_rank",
        (F.pmod(F.hash("userid", "windowid", "negative_number"), F.col("job_count")) + 1).cast("int")
    )

negative = negative_seed.join(jobs_indexed, ["windowid", "job_rank"]) \
    .select("userid", "jobid", "windowid", "split") \
    .join(
        positive.select("userid", "jobid", "windowid"),
        ["userid", "jobid", "windowid"],
        "left_anti"
    ) \
    .dropDuplicates() \
    .withColumn("label", F.lit(0.0))

negative.show(5)

+-------+------+--------+-----+-----+
| userid| jobid|windowid|split|label|
+-------+------+--------+-----+-----+
|1258502|594333|       1|Train|  0.0|
| 528317|526776|       1|Train|  0.0|
|1132853|612706|       1|Train|  0.0|
|  88931|579686|       1|Train|  0.0|
| 789199|113201|       1|Train|  0.0|
+-------+------+--------+-----+-----+
only showing top 5 rows



In [6]:
pairs = positive.unionByName(negative) \
    .dropDuplicates(["userid", "jobid", "windowid"])

pairs.groupBy("label").count().show()

+-----+-------+
|label|  count|
+-----+-------+
|  0.0| 963676|
|  1.0|1603111|
+-----+-------+



In [7]:
history_text = user_history.groupBy("userid", "windowid") \
    .agg(
        F.concat_ws(
            " ",
            F.collect_list(F.coalesce(F.col("jobtitle"), F.lit("")))
        ).alias("history_text")
    )

history_text.show(5)

+------+--------+--------------------+
|userid|windowid|        history_text|
+------+--------+--------------------+
|     7|       3|Customer Service ...|
|     9|       4|File Clerk Mechan...|
|    14|       4|Operations Manage...|
|    17|       3|Administrative As...|
|    23|       2| Assisting custom...|
+------+--------+--------------------+
only showing top 5 rows



In [8]:
users_prepared = users.withColumnRenamed("city", "user_city") \
    .withColumnRenamed("state", "user_state") \
    .withColumnRenamed("country", "user_country") \
    .withColumnRenamed("zipcode", "user_zipcode") \
    .withColumnRenamed("split", "user_split")

jobs_prepared = jobs.withColumnRenamed("city", "job_city") \
    .withColumnRenamed("state", "job_state") \
    .withColumnRenamed("country", "job_country") \
    .withColumnRenamed("zipcode", "job_zipcode")

In [9]:
data = pairs.join(users_prepared, ["userid", "windowid"], "left") \
    .join(jobs_prepared, ["jobid", "windowid"], "left") \
    .join(history_text, ["userid", "windowid"], "left")

data.select("userid", "jobid", "windowid", "split", "label", "title", "user_city", "job_city").show(5)

+------+------+--------+-----+-----+--------------------+---------+----------------+
|userid| jobid|windowid|split|label|               title|user_city|        job_city|
+------+------+--------+-----+-----+--------------------+---------+----------------+
|     7| 72945|       3|Train|  0.0| Benefits Specialist|  Roanoke|Sterling Heights|
|     9|136489|       4|Train|  1.0|UPS Part Time Pac...|  Houston|         Houston|
|     9|286530|       4|Train|  0.0|Network Administr...|  Houston|   Overland Park|
|     7|309823|       3|Train|  1.0|              Teller|  Roanoke|         Roanoke|
|     7|767113|       3|Train|  0.0|    Data Entry Clerk|  Roanoke|        Honolulu|
+------+------+--------+-----+-----+--------------------+---------+----------------+
only showing top 5 rows



In [10]:
data = data.withColumn(
    "same_city",
    F.when(
        F.lower(F.coalesce(F.col("user_city"), F.lit(""))) == F.lower(F.coalesce(F.col("job_city"), F.lit(""))),
        1.0
    ).otherwise(0.0)
)

data = data.withColumn(
    "same_state",
    F.when(
        F.lower(F.coalesce(F.col("user_state"), F.lit(""))) == F.lower(F.coalesce(F.col("job_state"), F.lit(""))),
        1.0
    ).otherwise(0.0)
)

data = data.withColumn(
    "same_country",
    F.when(
        F.lower(F.coalesce(F.col("user_country"), F.lit(""))) == F.lower(F.coalesce(F.col("job_country"), F.lit(""))),
        1.0
    ).otherwise(0.0)
)

data = data.withColumn(
    "job_duration_days",
    F.datediff(F.col("finish"), F.col("start")).cast("double")
)

data = data.withColumn(
    "text",
    F.concat_ws(
        " ",
        F.coalesce(F.col("title"), F.lit("")),
        F.coalesce(F.col("description"), F.lit("")),
        F.coalesce(F.col("requirements"), F.lit("")),
        F.coalesce(F.col("major"), F.lit("")),
        F.coalesce(F.col("history_text"), F.lit(""))
    )
)

In [11]:
categorical_cols = [
    "degree",
    "employed",
    "managerexp",
    "user_state",
    "job_state",
    "user_country",
    "job_country"
]

numeric_cols = [
    "jobcnt",
    "yearsexp",
    "emplcnt",
    "same_city",
    "same_state",
    "same_country",
    "job_duration_days"
]

for column in categorical_cols:
    data = data.withColumn(column, F.coalesce(F.col(column).cast("string"), F.lit("unknown")))

for column in numeric_cols:
    data = data.withColumn(column, F.coalesce(F.col(column).cast("double"), F.lit(0.0)))

In [12]:
train_raw = data.filter(F.col("split") == "Train")
test_raw = data.filter(F.col("split") == "Test")

print("train:", train_raw.count())
print("test:", test_raw.count())

train: 2341589


test: 225198


In [13]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, HashingTF, StringIndexer, OneHotEncoder, VectorAssembler

tokenizer = RegexTokenizer(
    inputCol="text",
    outputCol="words",
    pattern="\\W+",
    minTokenLength=2
)

hashing_tf = HashingTF(
    inputCol="words",
    outputCol="text_features",
    numFeatures=4096
)

indexers = [
    StringIndexer(
        inputCol=column,
        outputCol=column + "_index",
        handleInvalid="keep"
    )
    for column in categorical_cols
]

encoders = [
    OneHotEncoder(
        inputCol=column + "_index",
        outputCol=column + "_encoded"
    )
    for column in categorical_cols
]

assembler = VectorAssembler(
    inputCols=numeric_cols + [column + "_encoded" for column in categorical_cols] + ["text_features"],
    outputCol="features",
    handleInvalid="keep"
)

feature_pipeline = Pipeline(
    stages=[tokenizer, hashing_tf] + indexers + encoders + [assembler]
)

In [14]:
feature_model = feature_pipeline.fit(train_raw)

train_data = feature_model.transform(train_raw) \
    .select(
        "userid",
        "jobid",
        "windowid",
        "features",
        "label"
    )

test_data = feature_model.transform(test_raw) \
    .select(
        "userid",
        "jobid",
        "windowid",
        "features",
        "label",
        "user_state",
        "job_state",
        "degree",
        "yearsexp"
    )

train_data.show(5)
test_data.show(5)

26/05/09 13:40:45 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+-----+--------+--------------------+-----+
| userid|jobid|windowid|            features|label|
+-------+-----+--------+--------------------+-----+
| 636521|   17|       1|(4525,[0,1,5,6,8,...|  0.0|
|1300349|   69|       1|(4525,[0,1,4,5,6,...|  0.0|
| 161004|   69|       1|(4525,[0,1,4,5,6,...|  0.0|
|1059262|   69|       1|(4525,[0,1,5,6,8,...|  0.0|
| 871792|  277|       2|(4525,[0,1,5,6,8,...|  0.0|
+-------+-----+--------+--------------------+-----+
only showing top 5 rows



+------+-----+--------+--------------------+-----+----------+---------+-----------+--------+
|userid|jobid|windowid|            features|label|user_state|job_state|     degree|yearsexp|
+------+-----+--------+--------------------+-----+----------+---------+-----------+--------+
|345265|  426|       6|(4525,[0,1,2,4,5,...|  1.0|        TX|       TX|High School|     2.0|
|372636|  426|       6|(4525,[0,1,3,4,5,...|  1.0|        TX|       TX|High School|     2.0|
|497256|  426|       6|(4525,[0,1,3,4,5,...|  1.0|        TX|       TX|   Master's|     3.0|
|676799|  426|       6|(4525,[0,1,2,3,4,...|  1.0|        TX|       TX|       None|    19.0|
|866205|  426|       6|(4525,[0,1,3,4,5,...|  1.0|        TX|       TX|       None|     7.0|
+------+-----+--------+--------------------+-----+----------+---------+-----------+--------+
only showing top 5 rows



In [25]:
project_hdfs = "/user/team17/project/stage3"

train_data.select("features", "label") \
    .sample(False, 0.001, seed) \
    .write \
    .mode("overwrite") \
    .json(project_hdfs + "/data/train")

test_data.select("features", "label") \
    .sample(False, 0.001, seed) \
    .write \
    .mode("overwrite") \
    .json(project_hdfs + "/data/test")

In [26]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator

roc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

accuracy_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="accuracy"
)

f1_evaluator = MulticlassClassificationEvaluator(
    labelCol="label",
    predictionCol="prediction",
    metricName="f1"
)

In [30]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")

In [31]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    maxIter=20
)

lr_grid = ParamGridBuilder() \
    .addGrid(lr.regParam, [0.01]) \
    .addGrid(lr.elasticNetParam, [0.0, 0.5]) \
    .build()

lr_cv = CrossValidator(
    estimator=lr,
    estimatorParamMaps=lr_grid,
    evaluator=roc_evaluator,
    numFolds=3,
    parallelism=2,
    seed=seed
)

lr_cv_model = lr_cv.fit(train_data)

model1 = lr_cv_model.bestModel
predictions1 = model1.transform(test_data)

predictions1.select("label", "prediction", "probability").show(5)

+-----+----------+--------------------+
|label|prediction|         probability|
+-----+----------+--------------------+
|  1.0|       1.0|[0.06086855546049...|
|  0.0|       0.0|[0.92098955776398...|
|  1.0|       1.0|[0.02360176467356...|
|  0.0|       0.0|[0.99547547386390...|
|  0.0|       0.0|[0.89640904996469...|
+-----+----------+--------------------+
only showing top 5 rows



In [32]:
auc1 = roc_evaluator.evaluate(predictions1)
pr_auc1 = pr_evaluator.evaluate(predictions1)
accuracy1 = accuracy_evaluator.evaluate(predictions1)
f1_1 = f1_evaluator.evaluate(predictions1)

print("Logistic Regression")
print("AUC:", auc1)
print("PR AUC:", pr_auc1)
print("Accuracy:", accuracy1)
print("F1:", f1_1)

Logistic Regression
AUC: 0.950753556244805
PR AUC: 0.9881416394068768
Accuracy: 0.8644887410690101
F1: 0.8758816581909262


In [33]:
model1.write().overwrite().save(project_hdfs + "/models/model1")

predictions1.select("label", "prediction") \
    .coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(project_hdfs + "/output/model1_predictions")

evaluation = spark.createDataFrame(
    [
        ("model1_logistic_regression", float(auc1), float(pr_auc1), float(accuracy1), float(f1_1))
    ],
    ["model", "AUC", "PR_AUC", "Accuracy", "F1"]
)

evaluation.show(truncate=False)

evaluation.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(project_hdfs + "/output/evaluation")

+--------------------------+-----------------+------------------+------------------+------------------+
|model                     |AUC              |PR_AUC            |Accuracy          |F1                |
+--------------------------+-----------------+------------------+------------------+------------------+
|model1_logistic_regression|0.950753556244805|0.9881416394068768|0.8644887410690101|0.8758816581909262|
+--------------------------+-----------------+------------------+------------------+------------------+



In [34]:
dashboard_sample = predictions1.select(
    "userid",
    "jobid",
    "windowid",
    "label",
    "prediction",
    "user_state",
    "job_state",
    "degree",
    "yearsexp"
).sample(False, 0.02, seed).limit(10000)

dashboard_sample.coalesce(1) \
    .write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv(project_hdfs + "/output/dashboard_sample")

In [35]:
!hdfs dfs -ls /user/team17/project/stage3
!hdfs dfs -ls /user/team17/project/stage3/output
!hdfs dfs -ls /user/team17/project/stage3/models
!hdfs dfs -ls /user/team17/project/stage3/data

Found 3 items
drwxr-xr-x   - team17 students          0 2026-05-09 15:58 /user/team17/project/stage3/data
drwxr-xr-x   - team17 students          0 2026-05-09 19:42 /user/team17/project/stage3/models
drwxr-xr-x   - team17 students          0 2026-05-09 19:47 /user/team17/project/stage3/output
Found 3 items
drwxr-xr-x   - team17 students          0 2026-05-09 19:47 /user/team17/project/stage3/output/dashboard_sample
drwxr-xr-x   - team17 students          0 2026-05-09 19:45 /user/team17/project/stage3/output/evaluation
drwxr-xr-x   - team17 students          0 2026-05-09 19:45 /user/team17/project/stage3/output/model1_predictions
Found 1 items
drwxr-xr-x   - team17 students          0 2026-05-09 19:42 /user/team17/project/stage3/models/model1
Found 2 items
drwxr-xr-x   - team17 students          0 2026-05-09 15:58 /user/team17/project/stage3/data/test
drwxr-xr-x   - team17 students          0 2026-05-09 15:57 /user/team17/project/stage3/data/train


In [36]:
!mkdir -p ../output

!hdfs dfs -cat /user/team17/project/stage3/output/evaluation/*.csv > ../output/evaluation.csv
!hdfs dfs -cat /user/team17/project/stage3/output/model1_predictions/*.csv > ../output/model1_predictions.csv
!hdfs dfs -cat /user/team17/project/stage3/output/dashboard_sample/*.csv > ../output/dashboard_sample.csv